# Lab 4: Deterministic Agentic Workflows

Most useful “agentic” systems do not need a model to control every next step. When the process is known, Python should own the sequence, allowed branches, validation rules, and stopping conditions. The model can still help with bounded language tasks inside that structure.

We will build one workflow throughout the lab:

> A student reports that a project notebook stopped working after a package update. The demonstration is tomorrow. The system must prepare a grounded brief for a staff member—but it must not send a message, change the environment, or claim the problem is fixed.

That single example lets us see why a workflow needs typed state, prompt chaining, routing, parallel checks, validation, bounded revision, and tracing. Every new piece will solve a specific weakness in the previous version.

**Submission:** Complete the two exercises during class and upload the saved notebook with outputs to Canvas.

## Learning goals

By the end of the lab, you should be able to:

1. keep workflow control in code while using a model for bounded judgments;
2. explain why each stage exists and what failure it prevents;
3. route, validate, retry, and stop explicitly; and
4. test control flow separately from the quality of generated text.

## References

- [Anthropic: Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents)
- [Roitman (2026), Sections 18.5 and Chapter 20](https://arxiv.org/abs/2606.24937)
- [OpenAI: Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs)
- [Microsoft: Retry Pattern](https://learn.microsoft.com/en-us/azure/architecture/patterns/retry)

## The plan before we write code

We are building a small system that turns one student request into a safe staff brief.

### The student case

> “My project notebook stopped running after I updated a package. The demonstration is tomorrow, and I need safe next steps.”

The system is **not** allowed to repair the notebook, change packages, contact the student, or claim the problem is solved. Its only output is a brief that helps a staff member decide what to do next.

### The ingredients

| Ingredient | What it does in this case | Example result | Why we need it |
|---|---|---|---|
| **Shared state** | Carries the request and every intermediate result | Request text, route, facts, reviewer notes, brief, trace | Keeps stages from relying on hidden context |
| **Classifier** | Identifies the type and urgency of the request | `technology`, `high`, no mandatory human review | Gives later stages a bounded interpretation |
| **Router** | Converts the classification into an allowed destination | `technical_support` | Prevents a model label from directly authorizing arbitrary behavior |
| **Fact extractor** | Keeps only statements supported by the student’s text | Package was updated; notebook stopped; demo is tomorrow | Separates evidence from guesses such as “NumPy caused it” |
| **Diagnostic reviewer** | Suggests safe questions or checks that could narrow the issue | Ask for the error message; inspect the changed package version | Adds useful technical analysis without changing the environment |
| **Communication reviewer** | Suggests what the staff brief should explain clearly | Acknowledge urgency; distinguish facts from hypotheses | Keeps the brief understandable and appropriately cautious |
| **Aggregator** | Combines facts and reviewer notes into one typed brief | Summary, verified facts, proposed next steps, uncertainties | Produces one coherent artifact instead of pasted branch outputs |
| **Deterministic gate** | Enforces rules we can check exactly | Reject invented facts or “we fixed the notebook” | Blocks known violations cheaply and consistently |
| **Rubric evaluator** | Judges qualities that simple rules cannot fully capture | Is the brief grounded, clear, and cautious? | Adds a semantic quality check after hard rules pass |
| **Bounded reviser** | Revises a weak brief, but only within a fixed budget | At most two revisions | Improves the artifact without creating an endless loop |
| **Trace** | Records the path taken by this request | classify → route → facts → reviews → draft → evaluate | Lets us locate the first stage that went wrong |
| **Orchestrator** | Calls the ingredients in the permitted order | Stops early for human review or missing required analysis | Keeps control in Python rather than in the model |

### How we will assemble them

We will build the workflow in this order:

1. **Create the state** so every later stage has an explicit place to read and write.
2. **Add tracing** before adding model calls, so the workflow is observable from the beginning.
3. **Classify and extract facts** as two separate, typed judgments.
4. **Route with Python rules** so classifications map only to allowed destinations.
5. **Run the two independent reviewers** and define what happens if one fails.
6. **Aggregate their work** into a structured staff brief.
7. **Apply the deterministic gate** before spending another model call on evaluation.
8. **Evaluate and revise within a budget** so quality can improve without an unbounded loop.
9. **Package the stages in an orchestrator** and inspect the resulting trace.
10. **Test control flow separately from writing quality.**

This is the core pattern for the entire lab:

> Gather bounded evidence → apply specialized analysis → build one artifact → enforce hard rules → evaluate softer qualities → stop explicitly.

Keep this plan visible as you work. Each following section implements one ingredient and connects it to the pieces already built.

## 0. Setup

We need only three external libraries:

- **OpenAI** for bounded model calls;
- **Pydantic** for typed inputs and outputs; and
- **Pandas** to inspect the trace at the end.

The workflow remains local and synthetic. It produces a staff brief; it does not contact the student or change any system.

In [1]:
%pip install -q --upgrade openai pydantic pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 38.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.


In [2]:
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Literal

import pandas as pd
from openai import OpenAI
from pydantic import BaseModel, Field

try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = os.getenv("OPENAI_API_KEY")

MODEL = os.getenv("OPENAI_MODEL", "@30875-fall26-openai/gpt-5-nano")
client = OpenAI(api_key=api_key, base_url="https://api.portkey.ai/v1")

print("Model:", MODEL)
print("Client configured; the API key was not displayed.")

Model: @30875-fall26-openai/gpt-5-nano
Client configured; the API key was not displayed.


## 1. Start with the control decision

A deterministic workflow and an adaptive agent can both call models and tools. The important question is: **who chooses the next step?**

| | Deterministic workflow | Adaptive agent loop |
|---|---|---|
| Next step | Code or a declared graph | Model policy chooses dynamically |
| Allowed paths | Known in advance | May change after each observation |
| Stopping | Explicit terminal states | Model completion judgment plus hard limits |
| Best fit | Stable, repeatable processes | Tasks where the next action cannot be specified beforehand |

Our student-support process is mostly known: classify → route → extract facts → analyze → draft → validate. A model may produce different wording, but it cannot invent a new route or skip the validation gate. That is why a workflow is the simpler adequate architecture.

**Rationale:** We choose the control model before writing prompts. Otherwise, a collection of model calls can accidentally become an unbounded agent loop.

## 2. Give the workflow one shared state

Every stage needs a common record of what is known and what has already happened. Without a typed state, stages pass loosely structured dictionaries, fields silently disappear, and retries become hard to reason about.

For our example, the state separates:

- the original request;
- model-produced classification and facts;
- code-controlled route and status;
- intermediate reviewer notes;
- the final brief; and
- operational evidence such as calls, revisions, errors, and trace events.

**Rationale:** The state is the workflow’s memory and audit record. It prevents the final drafting stage from depending on hidden conversational context.

In [3]:
class RequestClassification(BaseModel):
    category: Literal["coursework", "technology", "administrative", "safety", "other"]
    urgency: Literal["low", "normal", "high"]
    needs_human_review: bool
    rationale: str = Field(min_length=5, max_length=250)


class ExtractedFacts(BaseModel):
    facts: list[str] = Field(min_length=1, max_length=6)


class StaffBrief(BaseModel):
    summary: str = Field(min_length=20, max_length=400)
    verified_facts: list[str] = Field(min_length=1, max_length=6)
    next_steps: list[str] = Field(min_length=1, max_length=4)
    uncertainties: list[str] = Field(default_factory=list, max_length=4)
    requires_human_review: bool


class WorkflowState(BaseModel):
    request_id: str
    request_text: str = Field(min_length=10)
    status: str = "created"
    classification: RequestClassification | None = None
    facts: list[str] = Field(default_factory=list)
    route: str | None = None
    reviewer_notes: dict[str, str] = Field(default_factory=dict)
    brief: StaffBrief | None = None
    quality_score: int | None = None
    model_calls: int = 0
    revisions: int = 0
    errors: list[str] = Field(default_factory=list)
    trace: list[dict] = Field(default_factory=list)


example = WorkflowState(
    request_id="REQ-0042",
    request_text=(
        "My project notebook stopped running after I updated a package. "
        "The demonstration is tomorrow, and I need safe next steps."
    ),
)
example.model_dump()

{'request_id': 'REQ-0042',
 'request_text': 'My project notebook stopped running after I updated a package. The demonstration is tomorrow, and I need safe next steps.',
 'status': 'created',
 'classification': None,
 'facts': [],
 'route': None,
 'reviewer_notes': {},
 'brief': None,
 'quality_score': None,
 'model_calls': 0,
 'revisions': 0,
 'errors': [],
 'trace': []}

## 3. Record transitions, not private reasoning

A useful trace answers: Which stage ran? What event occurred? What status followed? What small piece of metadata helps us debug the path?

The trace should not contain API keys, private records, or hidden model reasoning. For this request, recording the selected category or route is useful; storing the entire prompt again is not.

Each stage returns a copied state. This makes “before” and “after” states distinct, which is helpful in tests and bounded retries.

**Rationale:** If the final brief is wrong, we need to know whether the error began in classification, routing, fact extraction, analysis, or drafting.

In [4]:
def record(state: WorkflowState, stage: str, event: str, **details) -> WorkflowState:
    updated = state.model_copy(deep=True)
    updated.trace.append({
        "step": len(updated.trace) + 1,
        "stage": stage,
        "event": event,
        "status": updated.status,
        "details": details,
    })
    return updated


example = record(example, "initialize", "request_received", request_id=example.request_id)
pd.DataFrame(example.trace)

,step,stage,event,status,details
0,1,initialize,request_received,created,{'request_id': 'REQ-0042'}


## 4. Prompt chaining: separate two different judgments

One large prompt could classify the request, extract facts, recommend actions, and draft a reply. That would be difficult to inspect: if the result is wrong, which judgment failed?

We split the work into two bounded calls:

1. **Classify** the kind of request and whether it needs a person.
2. **Extract facts** that are explicitly stated in the request.

The extractor receives the classification for context, but it may not add causes or diagnoses. In our example, “the notebook stopped after a package update” is a fact; “NumPy is incompatible” is only a hypothesis.

**Rationale:** Chaining is useful when stages need different instructions and contracts. The trade-off is extra calls and the possibility that an early error affects later stages.

In [5]:
def classify_request(state: WorkflowState) -> WorkflowState:
    classification = client.responses.parse(
        model=MODEL,
        instructions=(
            "Classify this synthetic student-support request using only its text. "
            "Request human review for safety concerns or consequential ambiguity."
        ),
        input=state.request_text,
        text_format=RequestClassification,
    ).output_parsed

    updated = state.model_copy(deep=True)
    updated.classification = classification
    updated.status = "classified"
    updated.model_calls += 1
    return record(
        updated,
        "classify",
        "completed",
        category=classification.category,
        urgency=classification.urgency,
    )


def extract_facts(state: WorkflowState) -> WorkflowState:
    if state.classification is None:
        raise ValueError("Classify the request before extracting facts.")

    result = client.responses.parse(
        model=MODEL,
        instructions=(
            "Extract only explicit, decision-relevant facts. "
            "Do not infer a cause, identity, motive, diagnosis, or completed action."
        ),
        input=json.dumps({
            "request": state.request_text,
            "category": state.classification.category,
        }),
        text_format=ExtractedFacts,
    ).output_parsed

    updated = state.model_copy(deep=True)
    updated.facts = result.facts
    updated.status = "facts_extracted"
    updated.model_calls += 1
    return record(updated, "extract_facts", "completed", fact_count=len(result.facts))


chained = extract_facts(classify_request(example))
print(chained.classification)
print(chained.facts)

category='technology' urgency='high' needs_human_review=False rationale="User's notebook stops running after a package update and they need safe, step-by-step guidance before a presentation tomorrow. This is a technical issue related to software/environment; no explicit safety concerns."
['My project notebook stopped running after I updated a package.', 'The demonstration is tomorrow.', 'I need safe next steps.']


## 5. Route with code after the model classifies

The model supplies a classification; Python converts it into an allowed destination. These are different responsibilities.

For the notebook problem, a normal technology classification goes to `technical_support`. Safety concerns and any classification marked for human review override the ordinary route. Unknown categories also fail safely to a person.

**Rationale:** A probabilistic label should not directly authorize an action. The allowlist and precedence rules are deterministic, cheap to test, and visible to reviewers.

In [6]:
ROUTES = {
    "technology": "technical_support",
    "coursework": "coursework_support",
    "administrative": "administrative_support",
}


def choose_route(classification: RequestClassification) -> str:
    if classification.needs_human_review or classification.category == "safety":
        return "human_review"
    return ROUTES.get(classification.category, "human_review")


def route_request(state: WorkflowState) -> WorkflowState:
    if state.classification is None:
        raise ValueError("Cannot route an unclassified request.")

    updated = state.model_copy(deep=True)
    updated.route = choose_route(state.classification)
    updated.status = "routed"
    return record(updated, "route", "completed", route=updated.route)


routed = route_request(chained)
print("Selected route:", routed.route)

Selected route: technical_support


### Test the routing policy without spending model calls

These tests bypass the model and directly exercise the policy. If one fails, the problem is in our routing code, not in model behavior.

**Rationale:** Separating probabilistic classification tests from deterministic routing tests makes failures easier to locate.

In [7]:
routing_cases = [
    ("technology", False, "technical_support"),
    ("coursework", False, "coursework_support"),
    ("administrative", False, "administrative_support"),
    ("safety", True, "human_review"),
    ("technology", True, "human_review"),
    ("other", False, "human_review"),
]

for category, needs_review, expected in routing_cases:
    classification = RequestClassification(
        category=category,
        urgency="normal",
        needs_human_review=needs_review,
        rationale="Synthetic routing test case.",
    )
    assert choose_route(classification) == expected

print(f"Passed {len(routing_cases)} routing tests.")

Passed 6 routing tests.


## 6. Parallel checks: do independent work at the same time

Before drafting the brief, two reviewers inspect the same verified facts:

- **diagnostic:** proposes safe questions and checks that could narrow the problem;
- **communication:** suggests what a concise staff brief should explain.

Neither reviewer needs the other reviewer’s answer, so they can run concurrently. Parallelism may reduce elapsed time, but it does not reduce the number of calls and can increase rate-limit pressure.

For this example, diagnostic analysis is required; communication advice is optional. If the required branch fails, the workflow must stop rather than produce a weak technical brief.

**Rationale:** Parallelize only independent work, and decide the partial-failure policy before running the branches.

In [8]:
REVIEWERS = {
    "diagnostic": "Suggest safe diagnostic questions. Do not claim a cause or modify anything.",
    "communication": "Suggest what a concise and empathetic staff brief should explain.",
}
REQUIRED_REVIEWERS = {"diagnostic"}


def run_reviewer(name: str, instruction: str, state: WorkflowState) -> tuple[str, str]:
    response = client.responses.create(
        model=MODEL,
        instructions=instruction + " Use only the supplied facts. Return at most three bullets.",
        input=json.dumps({"route": state.route, "facts": state.facts}),
    )
    return name, response.output_text


def run_reviewers(state: WorkflowState) -> WorkflowState:
    notes, failures = {}, []
    started = time.perf_counter()

    with ThreadPoolExecutor(max_workers=2) as pool:
        jobs = {
            pool.submit(run_reviewer, name, instruction, state): name
            for name, instruction in REVIEWERS.items()
        }
        for job in as_completed(jobs):
            name = jobs[job]
            try:
                reviewer, text = job.result()
                notes[reviewer] = text
            except Exception as error:
                failures.append(name)

    updated = state.model_copy(deep=True)
    updated.reviewer_notes = notes
    updated.model_calls += len(notes)
    updated.errors.extend(f"Reviewer failed: {name}" for name in failures)

    missing_required = REQUIRED_REVIEWERS - notes.keys()
    updated.status = "human_review_required" if missing_required else "analysis_complete"
    return record(
        updated,
        "reviewers",
        "completed",
        completed=list(notes),
        failed=failures,
        elapsed_seconds=round(time.perf_counter() - started, 3),
    )


analyzed = run_reviewers(routed)
print("Completed:", list(analyzed.reviewer_notes))
print("Status:", analyzed.status)

Completed: ['communication', 'diagnostic']
Status: analysis_complete


## 7. Aggregate evidence into one constrained brief

Aggregation should reconcile inputs, not paste reviewer outputs together. The drafting call receives:

- the original request;
- the verified facts;
- the deterministic route; and
- the reviewer notes and any recorded failures.

The output is a `StaffBrief`, not free-form prose. Recommended next steps are proposals. The brief may say “check the package version,” but it may not say “we fixed the environment.”

**Rationale:** A structured aggregation contract prevents parallel branches from producing an incoherent or overconfident final artifact.

In [9]:
def draft_brief(state: WorkflowState) -> WorkflowState:
    if state.status == "human_review_required":
        return state

    brief = client.responses.parse(
        model=MODEL,
        instructions=(
            "Create a staff-facing brief. Preserve uncertainty. Copy verified_facts "
            "verbatim from the supplied list. Describe next steps as proposals, not completed actions."
        ),
        input=json.dumps({
            "request": state.request_text,
            "route": state.route,
            "verified_facts": state.facts,
            "reviewer_notes": state.reviewer_notes,
            "recorded_errors": state.errors,
        }),
        text_format=StaffBrief,
    ).output_parsed

    updated = state.model_copy(deep=True)
    updated.brief = brief
    updated.status = "drafted"
    updated.model_calls += 1
    return record(updated, "draft", "completed")


drafted = draft_brief(analyzed)
print(drafted.brief.model_dump_json(indent=2) if drafted.brief else drafted.status)

{
  "summary": "Support brief: The user's project notebook stopped running after a package update; a demonstration is tomorrow; they request safe next steps to restore functionality or provide a fallback demonstration.",
  "verified_facts": [
    "My project notebook stopped running after I updated a package.",
    "The demonstration is tomorrow.",
    "I need safe next steps."
  ],
  "next_steps": [
    "1) Identify the most recently updated package that could have caused the issue (check package manager history, environment spec, and recent installs).\n2) Attempt to reproduce the failure in a clean, isolated environment (e.g., create a new virtual environment or conda env with the same baseline specs).\n3) Consider a safe rollback or pinning strategy: revert the updated package to the previous known-good version or pin versions to a working baseline.\n4) Collect and review error messages, stack traces, and notebook/kernel logs to pinpoint the failure mode; request screenshots or log 

## 8. Add a deterministic gate before subjective evaluation

Some failures are known and machine-checkable. We do not need another model to detect them.

For this example, the gate checks that:

1. a brief exists;
2. every “verified fact” came from the extraction stage;
3. required human review was not removed; and
4. the brief does not claim that the system already sent, fixed, approved, or resolved something.

**Rationale:** Deterministic gates are fast, reproducible, and auditable. They should run before a more expensive model evaluator.

In [10]:
COMPLETION_CLAIMS = ("we sent", "we fixed", "we approved", "has been resolved")


def check_brief(state: WorkflowState) -> list[str]:
    if state.brief is None:
        return ["Brief is missing."]

    failures = []
    unsupported = set(state.brief.verified_facts) - set(state.facts)
    if unsupported:
        failures.append(f"Unsupported verified facts: {sorted(unsupported)}")

    if (
        state.classification
        and state.classification.needs_human_review
        and not state.brief.requires_human_review
    ):
        failures.append("Required human review was removed.")

    brief_text = state.brief.model_dump_json().lower()
    failures.extend(
        f"Prohibited completion claim: {phrase}"
        for phrase in COMPLETION_CLAIMS
        if phrase in brief_text
    )
    return failures


gate_failures = check_brief(drafted)
print("Gate passed:", not gate_failures)
print("Failures:", gate_failures)

Gate passed: True
Failures: []


## 9. Use a bounded evaluator–optimizer loop

The deterministic gate catches known violations, but it cannot reliably judge whether the brief is clear, appropriately cautious, and useful to staff. A model evaluator can assess those semantic qualities against a small rubric.

The evaluator is not an oracle. It may be inconsistent. Therefore:

- the deterministic gate remains authoritative for known rules;
- revision feedback cannot change upstream facts; and
- the workflow allows at most two revisions before stopping for human review.

**Rationale:** The loop exists to improve one artifact, not to give the model open-ended control. The revision budget guarantees termination.

In [11]:
class BriefEvaluation(BaseModel):
    grounded: bool
    clear: bool
    appropriately_cautious: bool
    feedback: list[str] = Field(default_factory=list, max_length=3)

    @property
    def score(self) -> int:
        return sum([self.grounded, self.clear, self.appropriately_cautious])


def evaluate_brief(state: WorkflowState) -> BriefEvaluation:
    return client.responses.parse(
        model=MODEL,
        instructions=(
            "Evaluate only these three criteria: grounded in verified facts, clear to staff, "
            "and appropriately cautious. Give short actionable feedback for failures."
        ),
        input=json.dumps({
            "verified_facts": state.facts,
            "brief": state.brief.model_dump() if state.brief else None,
        }),
        text_format=BriefEvaluation,
    ).output_parsed


def revise_brief(state: WorkflowState, feedback: list[str]) -> WorkflowState:
    revised = client.responses.parse(
        model=MODEL,
        instructions=(
            "Revise the brief using the feedback. Keep verified_facts exactly as supplied. "
            "Do not add facts, remove required review, or claim that an action occurred."
        ),
        input=json.dumps({
            "verified_facts": state.facts,
            "current_brief": state.brief.model_dump() if state.brief else None,
            "feedback": feedback,
        }),
        text_format=StaffBrief,
    ).output_parsed

    updated = state.model_copy(deep=True)
    updated.brief = revised
    updated.revisions += 1
    updated.model_calls += 1
    updated.status = "revised"
    return record(updated, "revise", "completed", revision=updated.revisions)


def improve_brief(
    state: WorkflowState,
    max_revisions: int = 2,
    evaluator=evaluate_brief,
    reviser=revise_brief,
) -> WorkflowState:
    current = state.model_copy(deep=True)

    for attempt in range(max_revisions + 1):
        feedback = check_brief(current)

        if not feedback:
            evaluation = evaluator(current)
            current.model_calls += 1
            current.quality_score = evaluation.score
            feedback = evaluation.feedback
            current = record(current, "evaluate", "completed", score=evaluation.score)

            if evaluation.score == 3:
                current.status = "approved"
                return current

        if attempt == max_revisions:
            current.status = "human_review_required"
            return record(current, "evaluate", "revision_budget_exhausted")

        current = reviser(current, feedback)

    return current  # Unreachable, but keeps the function's return explicit.


improved = improve_brief(drafted)
print("Status:", improved.status)
print("Score:", improved.quality_score)
print("Revisions:", improved.revisions)
print("Model calls:", improved.model_calls)

Status: approved
Score: 3
Revisions: 0
Model calls: 6


## 10. Make legal transitions visible

The workflow currently stores status as text. A small transition map documents which moves are legal. For example, a request cannot jump directly from `created` to `approved`, and `human_review_required` is terminal.

This map is intentionally smaller than a framework-based state machine. The goal is to make the policy readable before introducing graph libraries.

**Rationale:** Explicit transitions prevent accidental shortcuts when the workflow grows or is refactored.

In [12]:
LEGAL_NEXT = {
    "created": {"classified", "failed"},
    "classified": {"routed", "failed"},
    "routed": {"facts_extracted", "human_review_required", "failed"},
    "facts_extracted": {"analysis_complete", "human_review_required", "failed"},
    "analysis_complete": {"drafted", "failed"},
    "drafted": {"revised", "approved", "human_review_required", "failed"},
    "revised": {"revised", "approved", "human_review_required", "failed"},
    "approved": set(),
    "human_review_required": set(),
    "failed": set(),
}


def assert_transition(current: str, target: str) -> None:
    if target not in LEGAL_NEXT[current]:
        raise ValueError(f"Illegal transition: {current} -> {target}")


assert_transition("created", "classified")
try:
    assert_transition("created", "approved")
except ValueError as error:
    print("Blocked as expected:", error)

Blocked as expected: Illegal transition: created -> approved


## 11. See the whole workflow before packaging it

The graph below shows the paths the system permits. The trace will later show the one path taken by our notebook request.

```mermaid
flowchart LR
    A[Receive request] --> B[Classify]
    B --> C{Route}
    C -->|review required| H[Human review]
    C --> D[Extract facts]
    D --> E1[Diagnostic review]
    D --> E2[Communication review]
    E1 --> F[Draft brief]
    E2 --> F
    F --> G[Deterministic gate]
    G --> I[Rubric evaluator]
    I -->|pass| J[Approved brief]
    I -->|revise, budget remains| K[Revise]
    K --> G
    I -->|budget exhausted| H
```

**Rationale:** A graph communicates allowed structure; a trace provides evidence of actual execution. We need both.

## 12. Package the stages into one orchestrator

The orchestrator owns sequence and early exits. Notice how little “intelligence” it contains: its job is control.

For the example request, the expected path is technology classification → technical support → fact extraction → two reviews → draft → validation and bounded improvement. If classification or a required reviewer requests human help, the workflow stops safely.

**Rationale:** Packaging the workflow in one function gives callers a single entry point and makes termination behavior testable.

In [13]:
def run_workflow(request_id: str, request_text: str) -> WorkflowState:
    state = WorkflowState(request_id=request_id, request_text=request_text)
    state = record(state, "initialize", "request_received", request_id=request_id)

    try:
        state = classify_request(state)
        state = route_request(state)

        if state.route == "human_review":
            state.status = "human_review_required"
            return record(state, "route", "stopped_for_human_review")

        state = extract_facts(state)
        state = run_reviewers(state)
        if state.status == "human_review_required":
            return record(state, "reviewers", "required_reviewer_missing")

        state = draft_brief(state)
        return improve_brief(state)

    except Exception as error:
        state.errors.append(f"{type(error).__name__}: {error}")
        state.status = "failed"
        return record(state, "orchestrator", "exception", error_type=type(error).__name__)


complete_run = run_workflow(
    "REQ-0043",
    "My project notebook stopped running after I updated a package. "
    "The demonstration is tomorrow, and I need safe next steps.",
)

print("Final status:", complete_run.status)
print("Route:", complete_run.route)
print("Model calls:", complete_run.model_calls)
print("Errors:", complete_run.errors)

Final status: failed
Route: None
Model calls: 0
Errors: ['ValidationError: 1 validation error for RequestClassification\n  Invalid JSON: EOF while parsing a string at line 1 column 326 [type=json_invalid, input_value=\'{"category":"technology"... or ambiguous decisions\', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid']


## 13. Inspect the path that actually ran

The trace is more useful than a final status alone. It lets us confirm that the request followed the intended stages, stayed within the revision budget, and terminated.

If the brief is poor, start with the first suspicious trace event rather than immediately rewriting the final prompt.

**Rationale:** Debugging a workflow means locating the earliest bad transition or artifact, not treating the entire pipeline as one opaque model call.

In [14]:
trace_table = pd.json_normalize(complete_run.trace)
display(trace_table[["step", "stage", "event", "status"]])
print("Observed path:", " -> ".join(trace_table["stage"]))

,step,stage,event,status
0,1,initialize,request_received,created
1,2,orchestrator,exception,failed


Observed path: initialize -> orchestrator


## 14. Evaluate control flow separately from brief quality

A polished brief can come from the wrong path, and a correct path can still produce weak wording. Evaluate both.

1. **Component tests:** Does each function satisfy its input/output contract?
2. **Path tests:** Does each case select an allowed route and reach a safe terminal state?
3. **Artifact tests:** Is the final brief grounded, clear, cautious, and useful?

The cases below are deliberately small. First test routing with fixed classifications. Run the full workflow only when you want to spend model calls.

**Rationale:** Separating control evaluation from content evaluation tells us whether to fix Python policy, a prompt, a schema, or the test itself.

In [16]:
WORKFLOW_CASES = [
    {
        "id": "T1",
        "text": "My notebook reports a missing-package error and the lab is due Friday.",
        "expected_route": "technical_support",
    },
    {
        "id": "T2",
        "text": "I do not understand which report sections Project 1 requires.",
        "expected_route": "coursework_support",
    },
    {
        "id": "T3",
        "text": "This message describes an immediate safety concern requiring staff attention.",
        "expected_route": "human_review",
    },
]

pd.DataFrame(WORKFLOW_CASES)

# run workflow here and check final outcomes
for case in WORKFLOW_CASES:
    state = WorkflowState(request_id=case["id"], request_text=case["text"])
    state = record(state, "initialize", "request_received", request_id=case["id"])
    state = classify_request(state)
    state = route_request(state)
    state = extract_facts(state)
    state = run_reviewers(state)
    state = draft_brief(state)
    print(f"Case {case['id']}: {state.status}")
    # evaluate the accuracy
    if state.brief:
        evaluation = evaluate_brief(state)
    print (evaluation)


Case T1: drafted
grounded=True clear=True appropriately_cautious=True feedback=['No failures detected across grounding, clarity, and caution. For formal logs, optionally annotate which facts are verified and confirm the exact scope of ‘the lab due Friday’ to prevent ambiguity.']
Case T2: drafted
grounded=True clear=False appropriately_cautious=True feedback=['The brief contains long, run-on sentences and repetition. Break into concise bullets for readability.', 'Provide a concrete, enumerated plan staff can follow once the rubric is available (e.g., locate rubric, extract exact sections, map subsections).', "Trim redundancy (remove duplicate 'Proposal' items) and present uncertainties as a short, actionable note with a resolution plan."]
Case T3: drafted
grounded=True clear=True appropriately_cautious=True feedback=[]


In [17]:
df = pd.DataFrame(WORKFLOW_CASES)
df

,id,text,expected_route
0,T1,My notebook reports a missing-package error an...,technical_support
1,T2,I do not understand which report sections Proj...,coursework_support
2,T3,This message describes an immediate safety con...,human_review


## Exercise 1: Add a required policy reviewer

The current workflow requires diagnostic analysis but treats communication advice as optional. Extend it with a third reviewer named `policy`.

Your policy reviewer should check whether proposed next steps stay within this workflow’s authority: it may recommend checks, but it may not contact anyone, change software, grant an exception, or claim completion.

### Requirements

1. Add a narrowly scoped `policy` instruction to `REVIEWERS`.
2. Make `policy` a required reviewer.
3. Simulate both a successful policy review and a policy-review exception without waiting for a real API failure.
4. Verify that failure of the required branch produces `human_review_required`, while failure of the optional communication branch does not.
5. In 3–5 sentences, explain why this branch is a control safeguard but not proof of compliance with every real policy.

**Why this exercise:** You are changing the workflow’s failure policy, not merely adding another prompt.

In [ ]:
# Exercise 1 workspace
# Add the policy reviewer, then use a small deterministic mock or injected
# exception to test required-versus-optional branch failure behavior.

## Exercise 2: Prove that the revision loop terminates

Pass `improve_brief` a deterministic mock evaluator that always returns a failing evaluation and a mock reviser that only increments the revision count and records a trace event. Run the loop with a maximum of two revisions.

### Requirements

1. Pass the mock functions through the `evaluator` and `reviser` parameters; do not change the production functions.
2. Verify the terminal status is `human_review_required`.
3. Assert that the number of revisions never exceeds two.
4. Show the trace events that prove the loop stopped.
5. Propose one improvement that is better than simply increasing the retry limit.

**Why this exercise:** A bounded loop is a control guarantee. You should be able to demonstrate that guarantee without relying on the model to behave badly on demand.

In [ ]:
# Exercise 2 workspace
# Create a deterministic failing evaluator and verify the revision budget,
# terminal status, and trace evidence.

## Lab takeaways

- Use a deterministic workflow when the process and allowed paths are known.
- Keep shared state explicit so stages do not depend on hidden context.
- Split model calls only when the stages need different contracts or failure policies.
- Let models classify or transform; let code enforce routes, permissions, and limits.
- Parallelize independent work only after defining required and optional branches.
- Run deterministic gates before subjective model evaluators.
- Bound every revision loop and stop safely when the budget is exhausted.
- Use graphs to describe allowed paths and traces to inspect actual paths.
- Test routing and termination independently from the quality of generated prose.

The central design principle is simple: **models contribute judgments; the workflow retains control.**